# 1. Synthetic-stage purpose and research questions

The synthetic stage asks whether an externally imposed event effect can be detected; whether predictive direction and the known structural lag can be recovered; whether the event-response multiplier can be estimated; whether event-aware models improve volatility prediction; and how these results change with parameters, functionals, powered states and schedules.

The frozen convention is

\[
E_i \longrightarrow \omega_{i+1} \longrightarrow X_{i+1}.
\]

Event age 0 affects the first next-state observation; ages 0-12 correspond to direct predictive lags +1 through +13. Known-DGP quantities are retrospective synthetic references only.

# 2. Synthetic experiment map

The map distinguishes deterministic validation, one-seed illustration, repeated-seed robustness, and paired repeated-seed comparison. Notebook 06 supplies reusable infrastructure rather than a standalone scientific result.

In [19]:
from pathlib import Path
import json
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
paths = {
    '08': ROOT / 'notebook08_results' / 'notebook08_numeric_summary.csv',
    '09': ROOT / 'notebook09_results' / 'notebook09_calibration_numeric_summary.csv',
    '10': ROOT / 'notebook10_results' / 'stage4_seed_case_analysis.csv',
    '11metrics': ROOT / 'notebook11_results' / 'run_metrics.csv',
    '11config': ROOT / 'notebook11_results' / 'experiment_config.json',
    '12': ROOT / '12_log_state_event_decomposition.ipynb',
}
assert all(path.exists() for path in paths.values())
summary08 = pd.read_csv(paths['08'])
summary09 = pd.read_csv(paths['09'])
schedule_runs = pd.read_csv(paths['10'])
run_metrics = pd.read_csv(paths['11metrics'])
config11 = json.loads(paths['11config'].read_text(encoding='utf-8'))
nb12 = nbformat.read(paths['12'], as_version=4)
assert not any(output.get('output_type') == 'error' for cell in nb12.cells for output in cell.get('outputs', []))
schedule_summary = schedule_runs.groupby('case_name').agg(
    parametric_profile_rmse=('parametric_profile_RMSE','mean'),
    parametric_active_mae_gain=('active_sigma_parametric_MAE_gain_pct','median'),
    parametric_coverage_error=('parametric_absolute_coverage_error','mean'),
).reset_index()
assert set(schedule_summary.case_name) == {'dense_periodic','sparse_periodic','sparse_irregular'}
experiment_map = pd.DataFrame([
    ('00', 'Simulator, timing and information structure', 'Deterministic validation', 'final'),
    ('01-05', 'Benchmark detection, recovery and forecasting', 'Canonical one-path workflow', 'final'),
    ('06', 'Reusable workflow infrastructure', 'No standalone scientific result', 'final'),
    ('07', 'Nonlinear functional-form stress test', 'Exploratory one-seed evidence', 'final exploratory'),
    ('08', 'Scalar-parameter and replication robustness', '50-seed repeated evidence', 'final'),
    ('09', 'n=2 and scale-matched comparisons', 'Mixed illustrative and repeated evidence', 'final'),
    ('10', 'Event-frequency and schedule robustness', '50 paired seeds', 'final'),
    ('11', 'Dependence and causality comparison', '50-seed repeated comparison', 'final'),
    ('12', 'Log-state representation', 'Matched one-seed comparison', 'final exploratory'),
], columns=['notebook', 'main question', 'evidence type', 'status'])
display(experiment_map)

,notebook,main question,evidence type,status
0,00,"Simulator, timing and information structure",Deterministic validation,final
1,01-05,"Benchmark detection, recovery and forecasting",Canonical one-path workflow,final
2,06,Reusable workflow infrastructure,No standalone scientific result,final
3,07,Nonlinear functional-form stress test,Exploratory one-seed evidence,final exploratory
4,08,Scalar-parameter and replication robustness,50-seed repeated evidence,final
5,09,n=2 and scale-matched comparisons,Mixed illustrative and repeated evidence,final
6,10,Event-frequency and schedule robustness,50 paired seeds,final
7,11,Dependence and causality comparison,50-seed repeated comparison,final
8,12,Log-state representation,Matched one-seed comparison,final exploratory


# 3. Detection, direction and lag timing

GC provides linear directional predictive evidence when the distributed-lag model and history window are appropriate. A cumulative GC order \(p\) jointly contains lags \(1,\ldots,p\), so a selected order is not an isolated causal lag. Individual conditional-lag tests instead identify the strongest distinguishable direct lag.

TDMI describes unsigned nonlinear lagged dependence. Correlation and TDMI can look bidirectional because they do not condition on target history in the same way as GC or TE. Transfer entropy is directed conditional information transfer in principle but data-hungry and finite-sample sensitive. Detection is easier than structural-lag recovery; dense periodic calendars also create wider-lag aliases and reverse predictive artefacts. None of GC, TDMI, or TE proves interventionist structural causality.

The following rates pool the four non-null synthetic cases for the powered-state change target, giving 200 runs per method. Detection rates should be read alongside the held-out null false-positive rate, because a method that detects frequently may also over-detect under the no-effect system.

In [20]:
expected_cases = {'Dense periodic (n=1)', 'Sparse irregular (n=1)', 'State-dependent nu (n=1)', 'Locally scale-matched (n=2)', 'No effect (n=1)'}
methods = ['Lagged correlation', 'Granger causality p=1', 'Granger causality p=13', 'TDMI', 'Transfer entropy']
assert set(run_metrics.case_name) == expected_cases and set(run_metrics.method) == set(methods)
assert run_metrics.groupby(['case_name','method','transformation']).size().eq(50).all()
signal = run_metrics.loc[(run_metrics.transformation == 'changes') & (run_metrics.case_name != 'No effect (n=1)')]
held_out = set(config11['calibration_seed_split']['held_out'])
null_held_out = run_metrics.loc[(run_metrics.transformation == 'changes') & run_metrics.case_name.eq('No effect (n=1)') & run_metrics.seed.isin(held_out)]
assert len(signal) == 200 * len(methods) and null_held_out.groupby('method').size().eq(len(held_out)).all()
method_evidence = []
for method in methods:
    d = signal.loc[signal.method.eq(method)]
    null = null_held_out.loc[null_held_out.method.eq(method)]
    method_evidence.append({
        'method': method.replace('Granger causality ', 'GC ').replace('Transfer entropy', 'TE'),
        'forward detection rate': d.forward_detected.mean(),
        'forward-only classification rate': d.directional_classification.eq('forward only').mean(),
        'exact +1 recovery rate': np.nan if method == 'Granger causality p=13' else d.exact_plus_1_recovery.mean(),
        'direct-window recovery rate': d.direct_window_recovery.mean(),
        'held-out no-effect false-positive rate': null.forward_detected.mean(),
    })
method_evidence = pd.DataFrame(method_evidence)
rate_columns = [column for column in method_evidence if column.endswith('rate')]
display(method_evidence.style.format({column: '{:.1%}' for column in rate_columns}, na_rep='N/A'))
assert method_evidence.loc[method_evidence.method.eq('GC p=13'), 'exact +1 recovery rate'].isna().all()

,method,forward detection rate,forward-only classification rate,exact +1 recovery rate,direct-window recovery rate,held-out no-effect false-positive rate
0,Lagged correlation,95.0%,19.5%,52.0%,54.0%,8.0%
1,GC p=1,97.5%,57.5%,97.5%,97.5%,4.0%
2,GC p=13,100.0%,42.0%,N/A,100.0%,12.0%
3,TDMI,71.5%,14.0%,43.0%,41.0%,20.0%
4,TE,41.0%,4.5%,54.0%,41.0%,12.0%


Exact +1 recovery means that lag +1 is the strongest candidate within lags +1,…,+13. Direct-window recovery means that the overall selected positive-lag peak lies somewhere within that direct window. The held-out false-positive rates are based on 25 no-effect seeds per method and therefore have substantial sampling uncertainty

# 4. Event-profile recovery

The benchmark profile is deliberately smooth and exponential. Under that studied design, the parametric estimator generally benefits from the correctly specified shape, whereas the non-parametric estimator is more flexible but noisier. Sparse schedules reduce observations per event age and weaken recovery. Both estimates depend on the event-free background transition, and good forecasting does not by itself establish accurate structural-profile recovery.

No final executed Notebook 08 result supports an event-specific-weight claim, so none is made here.

# 5. Forecasting and calibration

Event-aware forecasts improve active-event-window performance in the synthetic benchmark. Event amplitude and volatility-innovation scale change signal-to-noise and forecast gains; sparse events generally reduce profile precision and improvement. Representative paths are pedagogical, whereas repeated-seed distributions are robustness evidence.

Forecasting is technically closer to a delayed-volatility nowcast: the next spot observation is available before the next volatility state is treated as observed. Calibration requires empirical coverage, coverage error, interval width and interval score together; 100% coverage alone is not necessarily good calibration. Forecast improvement does not guarantee event-profile recovery.

# 6. Robustness findings

Notebook 07 is a focused one-seed stress test, not general nonlinear robustness. Notebook 08 supplies repeated 50-seed parameter evidence: amplitude drives forecast gains, innovation scale governs signal-to-noise, and decay controls persistence. In Notebook 09, \(X_i=\sigma_i^2\) for \(n=2\) and the multiplier acts directly on the powered state; dedicated calibrations are not a pure test of \(n\), while scale-matched comparisons remain calibration-specific.

Notebook 10's corrected paired evidence shows that lower event frequency weakens forward-GC effect size, produces less precise event-profile recovery, and lowers active-window forecast gains. Lag 1 remains the modal strongest direct event lag across all three schedule cases. Dense periodic timing produces stronger wider-window TDMI dependence, consistent with periodic aliases. Once sparse event count is approximately held fixed, sparse irregular and sparse periodic schedules do not differ materially in profile recovery or forecasting.

Notebook 11 separates detection, directional classification and structural-lag recovery: GC p=1 matches the prespecified lag, GC p=13 is joint, TDMI/correlation detect dependence but struggle with direction, and TE is theoretically directional but weak in finite sparse samples. Notebook 12's conclusion is specific to its matched-feature linear implementation.

In [21]:
def metric_mean(frame, case, metric):
    value = frame.loc[(frame.case_name == case) & (frame.metric == metric), 'mean']
    assert len(value) == 1
    return float(value.iloc[0])

def schedule_value(case, column):
    value = schedule_summary.loc[schedule_summary.case_name.eq(case), column]
    assert len(value) == 1
    return float(value.iloc[0])

schedule_text = '; '.join(
    (
        f"{case.replace('_', ' ')}: "
        f"mean profile RMSE "
        f"{schedule_value(case, 'parametric_profile_rmse'):.5f}, "
        f"median active-window MAE gain "
        f"{schedule_value(case, 'parametric_active_mae_gain'):.1f}%"
    )
    for case in [
        'dense_periodic',
        'sparse_periodic',
        'sparse_irregular',
    ]
)
robustness = pd.DataFrame([
    ('07 nonlinear functionals', 'Frozen workflow remains usable in a focused stress test; numerical result omitted.', 'one-seed exploratory', 'not general nonlinear robustness'),
    ('08 scalar parameters', f"Repeated n=1 benchmark: active MAE gain {metric_mean(summary08,'benchmark','parametric_active_MAE_gain_pct'):.1f}%.", '50 seeds per case', 'chosen parameter grid'),
    (
    '09 powered states',
    (
        f"Dedicated n=2 benchmark mean active-window MAE gain "
        f"{metric_mean(summary09, 'n2_benchmark', 'parametric_active_MAE_gain_pct'):.1f}%."
    ),
    '50 seeds per case',
    (
        'Not a pure effect of changing n because the dedicated n=2 '
        'calibration also differs in scale and other parameters.'
    ),
),
    ('10 schedules', schedule_text + '. Final repeated analysis also reports modal direct lag 1 in every schedule case and stronger dense-periodic wider-window TDMI.', '50 paired seeds', 'three studied calendars'),
    ('11 methods', 'Detection, direction and structural-lag recovery are different tasks; no universal ranking.', '50 repeated seeds', 'method and representation dependence'),
    ('12 log state', 'Mostly inferior under the matched-feature raw linear predictor basis.', 'one-seed exploratory', 'not all log-scale models'),
], columns=['study','main conclusion','evidence strength','key limitation'])
display(robustness)
assert not schedule_summary.empty and all(schedule_value(case, 'parametric_profile_rmse') > 0 for case in schedule_summary.case_name)

,study,main conclusion,evidence strength,key limitation
0,07 nonlinear functionals,Frozen workflow remains usable in a focused st...,one-seed exploratory,not general nonlinear robustness
1,08 scalar parameters,Repeated n=1 benchmark: active MAE gain 22.8%.,50 seeds per case,chosen parameter grid
2,09 powered states,Dedicated n=2 benchmark mean active-window MAE...,50 seeds per case,Not a pure effect of changing n because the de...
3,10 schedules,"dense periodic: mean profile RMSE 0.00034, med...",50 paired seeds,three studied calendars
4,11 methods,"Detection, direction and structural-lag recove...",50 repeated seeds,method and representation dependence
5,12 log state,Mostly inferior under the matched-feature raw ...,one-seed exploratory,not all log-scale models


# 7. What the synthetic evidence establishes

1. Event-related predictive dependence can be detected under the studied synthetic design.
2. By construction, the first structural effect occurs at lag +1; the empirical methods are evaluated on whether they detect and recover that known timing.
3. Distributed persistence makes structural-lag recovery harder than detection.
4. A smooth parametric profile is advantageous when the true profile is exponential.
5. Event-aware forecasts can improve active-window prediction.
6. Amplitude, noise scale, event spacing and sample size materially affect performance.
7. Periodic schedules can generate aliases.
8. Repeated-seed evidence is needed to distinguish structural findings from path-specific noise.

# 8. What the synthetic evidence does not establish

It does not establish interventionist causality from GC, TDMI or TE alone; universally correct lag recovery; universal superiority of any dependence measure, \(n=2\), original-state models, or log-state models; or that good forecasts recover the true event mechanism.

Latent synthetic volatility is not observable ATM implied volatility. Empirically, neither the true event-free counterfactual transition nor the true multiplier is observed. One-seed results remain exploratory; repeated results remain conditional on the calibration. The benchmark profile and event timing are deliberately cleaner than real announcement and quote timing.

# 9. Synthetic-to-empirical transition

The synthetic powered-state estimand is

\[
\boxed{X_{t+1}=\omega_{t+1}^{n/2}F_t},\qquad X_t=\sigma_t^n,
\]

with the corresponding decomposition

\[
\boxed{\log X_{t+1}=\frac{n}{2}\log\omega_{t+1}+\log F_t}.
\]

Thus, if a synthetic powered-state coefficient satisfies \(\gamma_a=\frac n2\log\omega(a)\), its inversion is

\[
\boxed{\widehat\omega(a)=\exp\!\left(\frac{2\widehat\gamma_a}{n}\right)}.
\]

For \(n=1\), \(\widehat\omega(a)=\exp(2\widehat\gamma_a)\); for \(n=2\), \(\widehat\omega(a)=\exp(\widehat\gamma_a)\).

The empirical notebooks instead use observed ATM implied volatility. If

\[
\Delta\log IV_t=\text{background}+\gamma_aD_{t,a}+u_t,
\]

then \(\exp(\gamma_a)\) is the multiplicative change in the observed implied-volatility variable associated with that event-age coefficient. It is not automatically the simulator's structural \(\omega(a)\); mapping the empirical implied-volatility effect to the latent-state multiplier remains a methodological question to validate.

Latent physical volatility is not observed, ATM IV is not identical to the simulator state, hidden shocks and the true background transition are unavailable, the multiplier is unobserved, and empirical event/quote timing can be uncertain. The intended chain is

\[
\text{structural synthetic model} \rightarrow \text{observable-data estimator validated synthetically} \rightarrow \text{real FX-options application}.
\]

Ordinary \(\Delta IV_t\) remains an interpretable robustness specification. **Notebook 14 therefore begins with a real-data audit rather than immediately transferring the synthetic estimator.**